In [8]:
import os
import librosa
import numpy as np
from scipy.fftpack import dct
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pickle


####      경로 꼭 맞춰주기!!     ####
#모델 로드
with open('./model/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('./model/pca.pkl', 'rb') as f:
    pca = pickle.load(f)

with open('./model/model.pkl', 'rb') as f:
    svm_model = pickle.load(f)
    
    
# 테스트 데이터 경로
test_metadata_path = '../2501ml_data/label/test_label.txt'
test_data_path = '../2501ml_data/test'



# 노이즈 제거 함수
def spectral_subtract_denoise(y, sr, noise_duration=0.3, n_fft=2048, hop_length=512):
    D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    magnitude, phase = np.abs(D), np.angle(D)
    noise_frames = int((noise_duration * sr) / hop_length)
    noise_profile = np.mean(magnitude[:, :noise_frames], axis=1, keepdims=True)
    magnitude_denoised = np.maximum(magnitude - noise_profile, 0)
    D_denoised = magnitude_denoised * np.exp(1j * phase)
    y_denoised = librosa.istft(D_denoised, hop_length=hop_length)
    return y_denoised

# 특징 추출 함수
def extract_features(y_input):
    cqt = np.abs(librosa.cqt(y_input, sr=sr, n_bins=84, bins_per_octave=12))
    cqt_power = cqt ** 2
    log_cqt = np.log(cqt_power + 1e-8)
    cqcc = dct(log_cqt, type=2, axis=0, norm='ortho')[:19].T
    cqcc_T = cqcc.T
    cfcc_delta = librosa.feature.delta(cqcc_T)

    mfcc = librosa.feature.mfcc(y=y_input, sr=sr, n_mfcc=19, n_mels=128, fmin=40)
    mfcc_delta = librosa.feature.delta(mfcc, width=7)

    mfcc_combined = np.concatenate((cqcc_T, mfcc_delta, cfcc_delta), axis=0)
    mfcc_mean = np.mean(mfcc_combined, axis=1)
    return mfcc_mean

# 테스트 데이터 특징 추출
test_file_names = []
test_x = []
test_y = []

with open(test_metadata_path, 'r') as f:
    i = 0
    for line in f:
        print("\r", i, "개 테스트데이터 특징 추출중...", end="")
        i += 1
        spk, file_name, _, _, label = line.strip().split(' ')
        wav_path = os.path.join(test_data_path, file_name)
        y, sr = librosa.load(wav_path, sr=16000)
        y = spectral_subtract_denoise(y, sr)
        mfcc_mean = extract_features(y)
        test_x.append(mfcc_mean)
        test_y.append(label)
        test_file_names.append(file_name)
    print()

# numpy array 변환
test_x = np.array(test_x)
test_y = np.array(test_y)

# 테스트 데이터 전처리
test_x_scaled = scaler.transform(test_x)
test_x_reduced = pca.transform(test_x_scaled)

# 예측
predictions = svm_model.predict(test_x_reduced)
print("예측 결과:", predictions)

# 결과 저장
with open('./fak3killer_test_result.txt', 'w') as f:
    for i in range(len(predictions)):
        f.write(f"{test_file_names[i]} {predictions[i]}\n")

# 평가 스크립트 실행
!perl ../eval.pl ./fak3killer_test_result.txt ../2501ml_data/label/test_label.txt

 1999 개 테스트데이터 특징 추출중...
예측 결과: ['Real' 'Real' 'Real' ... 'Fake' 'Fake' 'Fake']
============ Results Analysis ===========
Test: ./fak3killer_test_result.txt
True: ../2501ml_data/label/test_label.txt
Accuracy: 98.15%
Hit: 1963, Total: 2000
